# Comparación en GPU (Colab, T4 gratis)

Corre **el mismo código del repo** en una GPU: `pipeline.py bench` transcribe con la rejilla de `config.yaml` (modelos × dispositivo × precisión) y guarda en cada corrida el dispositivo **leído de CTranslate2 en tiempo de ejecución** y el **pico de VRAM medido con NVML**.

Entorno → Cambiar tipo de entorno → **T4 GPU** antes de empezar.

In [ ]:
!nvidia-smi
!git clone https://github.com/luyov-belli/HW05_DS.git
%cd HW05_DS
!pip install -q -r requirements.txt

## Audio
Sube tus grabaciones a `MyDrive/HW05_audio/` con los nombres del guion (`E01.m4a`, …, `L05.m4a`). Si aún no grabas, la segunda línea genera el corpus sintético.

In [ ]:
CORPUS = "real"   # o "tts-es-PE"
if CORPUS == "real":
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p data/audio/real && cp /content/drive/MyDrive/HW05_audio/* data/audio/real/
else:
    !python pipeline.py tts --locales es-PE

In [ ]:
# Rejilla GPU/CPU/precisión + glosario con el modelo grande
!python pipeline.py bench --corpus {CORPUS}
# Escalera de las llamadas con el modelo grande en GPU
!python pipeline.py ladder --corpus {CORPUS}
!python pipeline.py transcribe --corpus {CORPUS} --model large-v3 --device cuda --task pedido --level L0 L1 L2 L3

In [ ]:
# Evidencia: dispositivo leído en runtime y VRAM medida de cada corrida
import json, glob
for f in sorted(glob.glob("results/transcripts/*.json")):
    m = json.load(open(f))["meta"]
    print(f"{m['model']:>9} {m['device_runtime']:>4} {m['compute_type_runtime']:>13} glos={m['glossary']!s:5} VRAM={m['vram_peak_mb']} MB  {m['seconds_per_audio_minute']} s/min  {m['gpu']}")

In [ ]:
# Descarga las transcripciones cacheadas y cópialas a results/transcripts/ de tu repo local
!zip -qr transcripts_gpu.zip results/transcripts results/ladder_*.json
from google.colab import files
files.download("transcripts_gpu.zip")